In [11]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [12]:
%load_ext autoreload
%autoreload 2

import sys, json
sys.path.append("..")
from functools import partial
from src.data import load_cuad
from src.split import split_contracts
from src.chunking import clause_chunks
from src.retrieval import make_dense_ranker
from src.bm25 import make_bm25_ranker
from src.hybrid import make_hybrid_ranker
from src.llm import complete
from src.generate import review_clause

contracts = load_cuad()
dev_idx, test_idx = split_contracts(contracts)
chunker = partial(clause_chunks, max_size=1500)
dense = make_dense_ranker(chunker, tag="clause-200-1500-200")
bm25 = make_bm25_ranker(chunker)
hybrid = make_hybrid_ranker([dense, bm25], weights=[1.0, 2.0])
print("ready")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
ready


In [13]:
i = next(i for i in dev_idx if contracts[i]["gold"]["Governing Law"])
c = contracts[i]
passages = hybrid(c, "Governing Law")[:3]

finding = review_clause(c["text"], "Governing Law", passages, complete)
print(c["title"])
print(json.dumps(finding.model_dump(), indent=2) if finding else "COULD NOT VERIFY, review manually")

LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGREEMENT
{
  "clause": "Governing Law",
  "found": true,
  "risk_level": "Medium",
  "explanation": "The agreement is governed by the laws of the State of Illinois, which may be the provider's home state rather than the customer's. Additionally, the clause does not specify a neutral jurisdiction or the customer's home jurisdiction, creating potential inconvenience in dispute resolution.",
  "cited_text": "6.9      Governing Law.  This Agreement is to be construed according to the laws          of the State of Illinois.",
  "suggested_redline": "This Agreement is governed by the laws of [Customer's jurisdiction]. The parties submit to the non-exclusive jurisdiction of the courts located in [Customer's jurisdiction].",
  "citation_start": 52036,
  "citation_end": 52151
}


In [14]:
absent = [i for i in dev_idx if not contracts[i]["gold"]["Governing Law"]][:3]

for i in absent:
    c = contracts[i]
    passages = hybrid(c, "Governing Law")[:3]
    f = review_clause(c["text"], "Governing Law", passages, complete)
    print(c["title"])
    if f is None:
        print("  COULD NOT VERIFY")
    elif not f.found:
        print("  found: False  (good)")
    else:
        print("  found: True  | risk:", f.risk_level)
        print("  quote:", f.cited_text[:200])
    print("-" * 60)

NELNETINC_04_08_2020-EX-1-JOINT FILING AGREEMENT
  found: False  (good)
------------------------------------------------------------
ThriventVariableInsuranceAccountB_20190701_N-6_EX-99.D(IV)_11720968_EX-99.D(IV)_Endorsement Agreement
  found: False  (good)
------------------------------------------------------------
Freecook_20180605_S-1_EX-10.3_11233807_EX-10.3_Hosting Agreement
  found: False  (good)
------------------------------------------------------------


In [15]:
c = contracts[dev_idx[0]]
top = hybrid(c, "Indemnification")[:3]
print(c["title"])
for ch in top:
    print(ch["start"], ch["end"])
    print(ch["text"][:400].replace("\n", " "))
    print("-" * 60)

LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGREEMENT
39859 41359
         5.3      Indemnification.  Company and Distributor  agree to indemnify,                   defend  and hold each other  harmless  from any and all suits,                   claims, obligations, liabilities, damages, losses and the like                   (including  attorneys'  fees and costs) relating to or arising                   out  of:  (A)  any  breach  of any  material  representati
------------------------------------------------------------
41159 41687
                  Distributor  agrees to  indemnify,  defend  and hold  harmless                   Company  from and  against  all  suits,  claims,  obligations,                   liabilities,   damages,  losses  and  the  like  (including  a                   attorneys'  fees  and  costs)  arising  out of or  related  to                   Distributor's  sales,   marketing  practices  or  unauthor
------------------------------------------------------------


In [16]:
import time
from src.pipeline import review_contract
from src.abstain import load_thresholds
from src.queries import CLAUSE_KEYWORDS

In [17]:

thresholds = load_thresholds()
signals = {
    "bm25": bm25,
    "dense_q": dense,
    "dense_kw": make_dense_ranker(chunker, tag="clause-200-1500-200", queries=CLAUSE_KEYWORDS),
}

c = contracts[dev_idx[0]]
t = time.time()
results = review_contract(c, hybrid, signals, thresholds, complete, pause=3)
print(c["title"], "|", round(time.time() - t, 1), "seconds")
for r in results:
    lawyer = bool(c["gold"][r.clause]) if r.clause in c["gold"] else "n/a"
    risk = r.finding.risk_level if r.finding and r.finding.found else "-"
    print(f"{r.clause:36} {r.status:11} by {r.decided_by:8} risk={risk:7} lawyer_present={lawyer}")

LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGREEMENT | 31.0 seconds
Governing Law                        found       by llm      risk=Medium  lawyer_present=True
Renewal Term                         not_found   by gate     risk=-       lawyer_present=True
Cap On Liability                     not_found   by llm      risk=-       lawyer_present=False
Uncapped Liability                   not_found   by gate     risk=-       lawyer_present=False
Termination For Convenience          not_found   by llm      risk=-       lawyer_present=False
Non-Compete                          not_found   by llm      risk=-       lawyer_present=False
Notice Period To Terminate Renewal   not_found   by llm      risk=-       lawyer_present=False
Indemnification                      found       by llm      risk=High    lawyer_present=n/a
